# 03 — Network-Aware QoS Adaptation Model

**Goal:** Train an ML model that predicts the optimal MQTT QoS level
given real-time network conditions and message priority.

**Two outputs:**
1. `network_state` classifier — predicts STABLE / DEGRADED / CRITICAL
2. `optimal_qos` classifier — predicts QoS 0/1/2 directly

Both plug into the Pi's SAAMP `setNetworkState()` or directly replace `decideAdaptiveQoS()`.

**Dataset:** Synthetic (generated) — replace with real Pi EMQX logs later.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.dpi'] = 120
print("Imports OK")

In [ ]:
from data_generator import generate_synthetic_dataset, describe_dataset
from feature_extraction import engineer_features, get_feature_cols

df = generate_synthetic_dataset(20000, save=True)
describe_dataset(df)

---
## Part A — Network State Predictor

Predicts whether the network will be STABLE/DEGRADED/CRITICAL
from current metrics.

In [ ]:
from network_predictor import label_network_state, preprocess

df_feat = preprocess(df)

FEATURE_COLS = [
    'rtt_avg', 'rtt_max', 'packet_loss', 'jitter',
    'queue_depth', 'queue_delta', 'throughput', 'bandwidth',
    'drop_rate', 'bridge_alive', 'clients',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'priority_scaled', 'bw_rtt_ratio', 'loss_rtt_product',
]

X = df_feat[FEATURE_COLS].fillna(0)
y = df_feat['network_state_enc']

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Train shape:", X_train.shape)
print("Test shape:",  X_test.shape)
print("\nNetwork state distribution in train:")
state_names = {0: 'STABLE', 1: 'DEGRADED', 2: 'CRITICAL'}
print(pd.Series(y_train).map(state_names).value_counts())

In [ ]:
MODELS = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=12,
                                            class_weight='balanced', random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                                    random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE, verbose=-1),
}

results = []
best_model = None
best_score = 0
best_name  = ''

for name, model in MODELS.items():
    use_scaled = name == 'LogisticRegression'
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test

    model.fit(Xtr, y_train)
    preds = model.predict(Xte)

    acc = accuracy_score(y_test, preds)
    results.append({'Model': name, 'Accuracy': acc})

    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=list(state_names.values())))

    if acc > best_score:
        best_score = acc
        best_model = model
        best_name  = name

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(f"\n{'='*50}")
print(f"Best: {best_name} (Accuracy: {best_score:.4f})")
display(results_df)

In [ ]:
_, ax = plt.subplots(figsize=(8, 4))
colors = ['#27ae60' if i == 0 else '#95a5a6' for i in range(len(results_df))]
ax.barh(results_df['Model'], results_df['Accuracy'], color=colors)
ax.set_xlabel('Accuracy')
ax.set_title('Network State Prediction — Model Comparison')
ax.set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_best = best_model.predict(X_test_sc if best_name == 'LogisticRegression' else X_test)
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=list(state_names.values()))
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix — {best_name}')
plt.grid(False)
plt.show()

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    imp.plot(kind='barh', color='coral')
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## Part B — Direct QoS Adaptation Classifier

Instead of predicting network state and using rules, this model
predicts the optimal QoS level (0/1/2) directly from features + priority.

In [ ]:
y_qos = df_feat['optimal_qos']
mask_ok = y_qos >= 0

X_qos = X[mask_ok]
y_qos = y_qos[mask_ok]

print("Optimal QoS distribution:")
print(y_qos.value_counts().sort_index())
print(f"\nTotal trainable samples: {len(X_qos)}")
print(f"Dropped (always-buffer): {(~mask_ok).sum()}")

In [ ]:
split_q = int(len(X_qos) * 0.8)
Xq_train, Xq_test = X_qos.iloc[:split_q], X_qos.iloc[split_q:]
yq_train, yq_test = y_qos.iloc[:split_q], y_qos.iloc[split_q:]

scaler_q = StandardScaler()
Xq_train_sc = scaler_q.fit_transform(Xq_train)
Xq_test_sc  = scaler_q.transform(Xq_test)

qos_models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10,
                                            random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6,
                               random_state=RANDOM_STATE, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=200, max_depth=6,
                                random_state=RANDOM_STATE, verbose=-1),
}

qos_results = []
best_qos_model = None
best_qos_score = 0

for name, model in qos_models.items():
    model.fit(Xq_train, yq_train)
    preds = model.predict(Xq_test)
    acc = accuracy_score(yq_test, preds)
    qos_results.append({'Model': name, 'Accuracy': acc})
    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(classification_report(yq_test, preds))
    if acc > best_qos_score:
        best_qos_score = acc
        best_qos_model = model
        best_qos_name  = name

print(f"\nBest QoS model: {best_qos_name} ({best_qos_score:.4f})")

In [ ]:
yq_pred = best_qos_model.predict(Xq_test)
cm_q = confusion_matrix(yq_test, yq_pred)
disp_q = ConfusionMatrixDisplay(cm_q, display_labels=['QoS 0', 'QoS 1', 'QoS 2'])
disp_q.plot(cmap='Greens', values_format='d')
plt.title(f'QoS Adaptation Confusion Matrix — {best_qos_name}')
plt.grid(False)
plt.show()

In [ ]:
if hasattr(best_qos_model, 'feature_importances_'):
    imp_q = pd.Series(best_qos_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    imp_q.plot(kind='barh', color='teal')
    plt.title(f'Feature Importance — QoS Adaptation ({best_qos_name})')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## Save Models

In [ ]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

# Network state predictor (for use with existing SAAMP decideAdaptiveQoS)
joblib.dump(best_model, MODELS_DIR / 'network_model.pkl')
joblib.dump(scaler,     MODELS_DIR / 'network_scaler.pkl')
joblib.dump(FEATURE_COLS, MODELS_DIR / 'network_features.pkl')

meta = {
    'model_type':  best_name,
    'task':        'network_state_prediction',
    'features':    FEATURE_COLS,
    'accuracy':    float(best_score),
    'state_map':   {0: 'STABLE', 1: 'DEGRADED', 2: 'CRITICAL'},
}
(MODELS_DIR / 'network_model_meta.json').write_text(json.dumps(meta, indent=2))

# Direct QoS adaptation model (bypasses rules, predicts QoS directly)
joblib.dump(best_qos_model, MODELS_DIR / 'qos_model.pkl')
joblib.dump(scaler_q,       MODELS_DIR / 'qos_scaler.pkl')

qos_meta = {
    'model_type':  best_qos_name,
    'task':        'qos_adaptation',
    'features':    FEATURE_COLS,
    'accuracy':    float(best_qos_score),
    'qos_classes': [0, 1, 2],
}
(MODELS_DIR / 'qos_model_meta.json').write_text(json.dumps(qos_meta, indent=2))

print("Saved models:")
print(f"  Network state predictor:  {best_name}")
print(f"  QoS adaptation model:     {best_qos_name}")
print(f"\nFiles in {MODELS_DIR}:")
for f in sorted(MODELS_DIR.iterdir()):
    print(f"  {f.name}")

---
## Next: Deploy to Pi

1. Copy `network_model.pkl`, `network_scaler.pkl`, `network_features.pkl`
   to the Raspberry Pi backend.

2. In `saampMqtt.js`, replace `getNetworkState()` with the ML prediction:
   - Run Python script from Node, OR
   - POST the prediction to `/api/network/state`, OR
   - Use `onnxruntime` to run the model directly in Node.js.

3. The existing `decideAdaptiveQoS()` uses the network state — no changes needed.